# Spatial 原始数据体量检查（本地）

数据根目录默认 **`D:\pretrain\Spatial\data`**。

- **统计对象**：除常见 **label / mask / seg** 路径与文件名以外的文件（启发式过滤）。
- **不统计**：所有 **`.npy`**（预处理结果不要求体量巡检）。
- **其余后缀**均会按扩展名汇总个数与体积（如 `.nii`、`.nii.gz`、`.mha`、`.dcm`、`.zip`、无扩展名等）。
- 不解析文件内容，仅 `os.walk` + 扩展名 + `stat`。数据量很大时也只做一遍遍历。

In [ ]:
from __future__ import annotations

import os
from collections import defaultdict
from pathlib import Path

DATA_ROOT = Path(r"D:/pretrain/Spatial/data")


def path_has_label_part(p: Path) -> bool:
    exact_names = {
        "label", "labels", "mask", "masks", "seg", "segs",
        "segmentation", "segmentations", "gt", "gts",
        "ground_truth", "ground-truth", "annotation", "annotations",
        "lesion_annotation",
    }
    substrings_long = (
        "label", "mask", "segmentation", "annotation", "ground_truth", "ground-truth",
    )
    for part in (s.lower() for s in p.parts):
        stem = part.rsplit(".", 1)[0] if "." in part else part
        if part in exact_names or stem in exact_names:
            return True
        for tok in ("seg", "mask"):
            if part == tok or part.startswith(tok + "-") or part.endswith("-" + tok):
                return True
            if "-" + tok + "-" in part or "_" + tok + "_" in part:
                return True
        if stem in ("gt", "gts") or stem.endswith("_gt") or stem.endswith("-gt"):
            return True
        if any(t in part for t in substrings_long if len(t) >= 5):
            return True
    name = p.name.lower()
    substrings = (
        "-label", "_label", ".label.", "label.nii", "label.mha", "label.mhd",
        "-seg.", "_seg.", ".seg.", "-seg_", "_seg-",
        "seg.nii", "seg.mha",
        "-mask", "_mask", ".mask.", "mask.nii",
        "_gt.", "-gt.", "_gt_", "-gt-",
    )
    return any(s in name for s in substrings)


def iter_raw_files(root: Path):
    """非 label 分支下的文件；跳过 .npy。"""
    root = root.resolve()
    if not root.is_dir():
        return
    for dirpath, dirnames, filenames in os.walk(root):
        dpath = Path(dirpath)
        if path_has_label_part(dpath):
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not path_has_label_part(dpath / d)]
        for fn in filenames:
            fpath = dpath / fn
            if path_has_label_part(fpath):
                continue
            if fpath.suffix.lower() == ".npy":
                continue
            if fpath.is_file():
                yield fpath


def ext_key(fp: Path) -> str:
    s = fp.suffix.lower()
    if s == ".gz" and fp.name.lower().endswith(".nii.gz"):
        return ".nii.gz"
    return s or "(no ext)"


def summarize(data_root: Path) -> None:
    data_root = data_root.resolve()
    if not data_root.is_dir():
        print(f"路径不存在或不是目录: {data_root}")
        return

    sets = sorted([p for p in data_root.iterdir() if p.is_dir()])
    print(f"根: {data_root}  |  顶层数据集数: {len(sets)}\n")

    for ds in sets:
        by_ext: dict[str, int] = defaultdict(int)
        bytes_by: dict[str, int] = defaultdict(int)
        n = 0
        for fp in iter_raw_files(ds):
            n += 1
            k = ext_key(fp)
            by_ext[k] += 1
            try:
                bytes_by[k] += fp.stat().st_size
            except OSError:
                pass
        print("=" * 56)
        print(ds.name)
        print(f"  原始文件数（已排除 label 规则 + 全部 .npy）: {n}")
        if not by_ext:
            print("  （无文件）\n")
            continue
        for k in sorted(by_ext, key=lambda x: (-by_ext[x], x)):
            gib = bytes_by[k] / (1024 ** 3)
            print(f"    {k:12s}  {by_ext[k]:8d}  个  ~{gib:10.2f} GiB")
        print()


summarize(DATA_ROOT)